# Ingest races file

In [0]:
dbutils.widgets.text('p_batch_id', '')

In [0]:
v_batch_id = dbutils.widgets.get('p_batch_id')
v_batch_id

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/races.csv'

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType()),
])

In [0]:
races_df = (
    spark.read
        .format('csv')
        .option('header', True)
        .schema(races_schema)
        .load(source_file)
)

In [0]:
display(races_df)

In [0]:
races_df_final = add_ingestion_metadata(races_df)

In [0]:
display(races_df_final)

In [0]:
%sql
DROP TABLE formula1.bronze.races

In [0]:
write_to_bronze (
    final_df = races_df_final,
    target_table = f'{catalog_name}.{bronze_schema}.races',
    batch_id = v_batch_id
)

In [0]:
# (
#     races_df_final.write
#         .mode('overwrite')
#         .format('delta')
#         .saveAsTable(f'{catalog_name}.{bronze_schema}.races')
# )

In [0]:
%sql
SELECT * FROM formula1.bronze.races;